# Find Candidate Induction Heads

Using the head embedding distances from notebook 03/04, find heads in other models
(pythia-1b, gemma-2b, etc.) that are nearby known GPT-2 small induction heads.

These candidates are then tested via ablation in notebook 06.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import numpy as np

# attention-motifs
from attention_motifs.attnpedia.attnpedia import AttentionPedia
from attention_motifs.features.analysis import DistanceTensorResult
from attention_motifs.ablation.candidates import (
    CandidateHeads,
    get_known_induction_heads,
    find_candidate_induction_heads,
    get_control_heads,
    save_candidates_to_file,
)

In [2]:
# config
pl.Config.set_tbl_rows(20)
PATH_BASE: Path = Path("../data/")

# Load distances and known heads

In [3]:
# load head distances
HEAD_DISTS: DistanceTensorResult = DistanceTensorResult.read_raw(
    PATH_BASE / "features" / "head_dists_raw",
)
print(f"Loaded distances for {HEAD_DISTS.n_heads} heads")
print(f"Models: {set(h.split(':')[0] for h in HEAD_DISTS.cls_values)}")

Loaded distances for 1520 heads
Models: {'gemma-2b', 'gpt2-small', 'pythia-1b', 'Llama-3-2-1B', 'gpt2-medium', 'gemma-2-2b'}


In [ ]:
# load known induction heads
ATTNPEDIA: AttentionPedia = AttentionPedia()
KNOWN_INDUCTION: list[str] = get_known_induction_heads(ATTNPEDIA)
print(f"Known induction heads ({len(KNOWN_INDUCTION)}):")
for head in KNOWN_INDUCTION:
    print(f"  {head}")

# Find candidate induction heads in other models

In [ ]:
# find candidates based on embedding proximity
CANDIDATES: CandidateHeads = find_candidate_induction_heads(
    HEAD_DISTS,
    reference_heads=KNOWN_INDUCTION,
    k_neighbors=20,
    exclude_reference_model=True,
    score_method="frequency",
)

print(f"Reference heads used: {len(CANDIDATES.reference_heads)}")
print(f"Models with candidates: {list(CANDIDATES.candidates_by_model.keys())}")

In [6]:
# show top candidates per model
for model, candidates in CANDIDATES.candidates_by_model.items():
    print(f"\n{model}:")
    for head, score in candidates[:10]:
        print(f"  {head}: {score:.3f}")


gpt2-medium:
  gpt2-medium:L12:H1: 3.450
  gpt2-medium:L11:H1: 3.100
  gpt2-medium:L6:H1: 3.000
  gpt2-medium:L9:H9: 2.650
  gpt2-medium:L13:H14: 2.600
  gpt2-medium:L10:H1: 2.300
  gpt2-medium:L7:H2: 1.650
  gpt2-medium:L10:H8: 1.550
  gpt2-medium:L9:H3: 1.000
  gpt2-medium:L6:H15: 1.000

gemma-2-2b:
  gemma-2-2b:L6:H3: 2.150
  gemma-2-2b:L15:H0: 1.050
  gemma-2-2b:L6:H2: 0.900
  gemma-2-2b:L7:H6: 0.800
  gemma-2-2b:L8:H6: 0.500
  gemma-2-2b:L8:H1: 0.300
  gemma-2-2b:L22:H6: 0.250

Llama-3-2-1B:
  Llama-3-2-1B:L5:H10: 2.150
  Llama-3-2-1B:L10:H23: 1.950
  Llama-3-2-1B:L2:H20: 1.400
  Llama-3-2-1B:L3:H7: 0.900
  Llama-3-2-1B:L5:H9: 0.900
  Llama-3-2-1B:L5:H11: 0.850
  Llama-3-2-1B:L3:H6: 0.850
  Llama-3-2-1B:L4:H18: 0.800
  Llama-3-2-1B:L5:H8: 0.800
  Llama-3-2-1B:L3:H10: 0.750

pythia-1b:
  pythia-1b:L7:H1: 1.700
  pythia-1b:L5:H7: 0.850
  pythia-1b:L4:H4: 0.800
  pythia-1b:L7:H2: 0.750
  pythia-1b:L5:H4: 0.600
  pythia-1b:L7:H0: 0.550
  pythia-1b:L6:H2: 0.450
  pythia-1b:L11:H7: 0.4

In [ ]:
# convert to DataFrame for analysis
CANDIDATES_DF: pl.DataFrame = CANDIDATES.to_dataframe()
CANDIDATES_DF

# Visualize candidate distribution

In [ ]:
# plot score distribution by model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# histogram of scores
for model in CANDIDATES.candidates_by_model.keys():
    model_df = CANDIDATES_DF.filter(pl.col("model") == model)
    axes[0].hist(model_df["score"].to_numpy(), alpha=0.5, label=model, bins=20)

axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Candidate Score Distribution by Model")
axes[0].legend()

# layer distribution of top candidates
TOP_N: int = 10
for model in CANDIDATES.candidates_by_model.keys():
    top_df = CANDIDATES_DF.filter(pl.col("model") == model).head(TOP_N)
    layers = top_df["layer"].to_numpy()
    axes[1].hist(layers, alpha=0.5, label=model, bins=range(0, 20))

axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Layer Distribution of Top {TOP_N} Candidates")
axes[1].legend()

plt.tight_layout()
Path("figures").mkdir(exist_ok=True)
plt.savefig("figures/ablation_candidates.pdf", bbox_inches="tight")

# Analyze nearest neighbors for each reference head

In [ ]:
# show which candidates appear for multiple reference heads
candidate_counts: Counter[str] = Counter()
for ref_head, neighbors in CANDIDATES.all_neighbors.items():
    for neighbor, dist in neighbors:
        candidate_counts[neighbor] += 1

print("Heads appearing in top-K for multiple reference heads:")
for head, count in candidate_counts.most_common(20):
    if count > 1:
        print(f"  {head}: appears {count} times")

# Get control heads for baseline comparison

In [ ]:
# get control heads (far from induction heads) for each model
CONTROL_HEADS: dict[str, list[str]] = {}
for model in CANDIDATES.candidates_by_model.keys():
    controls = get_control_heads(
        HEAD_DISTS, CANDIDATES, model, n_controls=5, method="far"
    )
    CONTROL_HEADS[model] = controls
    print(f"\n{model} control heads (far from induction):")
    for head in controls:
        print(f"  {head}")

# Save candidates for ablation study

In [ ]:
# save candidates to file
output_path: Path = PATH_BASE / "ablation" / "candidates.json"
save_candidates_to_file(CANDIDATES, output_path)
print(f"Saved candidates to {output_path}")

# also save as CSV for easy viewing
CANDIDATES_DF.write_csv(PATH_BASE / "ablation" / "candidates.csv")